# Hybrid RAG for Personalized Skincare Recommendations

This notebook builds and evaluates a **Hybrid RAG** recommender that combines:
- **Vector search** over review semantics (baseline)
- **Graph signals** from a Neo4j Knowledge Graph (brand loyalty re-ranking)

Outputs:
- Side-by-side **Vector-only vs Hybrid** Top-K recommendations
- A short **rationale** explaining why the Hybrid ranking changed
- Quantitative (Hit Rate@10, NDCG@10) + Qualitative (Ragas) evaluation.

## Quickstart

1. Create a `.env` file from `.env.example` (recommended).  
   If not set, the notebook will prompt via `getpass()` at runtime.
2. Ensure `./data` contains:
   - `amazon_face_core_meta.csv`
   - `amazon_face_core_reviews.csv`
3. Run cells top-to-bottom.

## Method

### Data
Subset of **Amazon Reviews 2023** (Facial Skincare), filtered to "core users" with sufficient purchases.

### Algorithm
1. **Candidate Generation:** Vector search retrieves top-N review-based candidates.
2. **Dynamic Re-ranking:** Apply a **log-weighted boost** based on user-brand history in the KG.

**Scoring:**
\[
FinalScore = VectorScore \times (1 + \log(1 + HistoryCount) \times w)
\]


## Problem & Approach

**Problem:** Vector search retrieves semantically similar products, but does not reflect user-specific preferences (e.g., brand loyalty).  
**Solution:** Re-rank vector candidates by injecting a **log-weighted brand loyalty signal** from a Neo4j Knowledge Graph.

**Core idea:** Use vector search for candidate generation, then apply a graph-based boost as a tie-breaker for ambiguous queries.

---
*See below for detailed implementation and code execution
* Note: Some metrics and example rankings may vary slightly between runs due to random sampling and LLM-based query paraphrasing.
For the primary reported results, refer to the exported files in `reports/`.*

## Technical Implementation
### Step 1. Environment & Credentials
- Load environment variables (`.env`) or prompt via `getpass()`

In [1]:
# ================================================
# Execution Setting (⚠️IMPORTANT: READ ME FIRST)
# ================================================

RUN_RAGAS = True  # High cost (LLM-based eval). Change to True ONLY when generating the final report.

In [2]:
# ==========================================
# Library
# ==========================================

# Standard library
import os
import random
import time
import sys
import json
from datetime import datetime
from dataclasses import asdict, is_dataclass
from getpass import getpass
from pathlib import Path
import traceback

# Third-party
import numpy as np
import pandas as pd
from tqdm import tqdm

from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_similarity,
)
from datasets import Dataset

# Make src importable (notebook-friendly)
repo_root = Path.cwd()
# If the notebook is executed from notebooks/, move to repo root
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [3]:
# =====================================================
# Global configuration
# =====================================================

load_dotenv()  # Loads .env from the current working directory (if present)

# Secrets: prefer env vars; fallback to interactive prompt
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("Enter OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # For libraries that read from env

# Models: env override supported
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
JUDGE_MODEL = os.getenv("JUDGE_MODEL", "gpt-4.1-mini")

# Clients (kept consistent with your current notebook usage)
client = OpenAI(api_key=OPENAI_API_KEY)
llm_service = ChatOpenAI(model=CHAT_MODEL)
llm_judge = ChatOpenAI(model=JUDGE_MODEL)
emb = OpenAIEmbeddings(model=EMBEDDING_MODEL)

print(f"Models configured: service={CHAT_MODEL}, judge={JUDGE_MODEL}, embed={EMBEDDING_MODEL}")

Models configured: service=gpt-4.1-mini, judge=gpt-4.1-mini, embed=text-embedding-3-small


In [4]:
# ============================================================
# Neo4j connection (Global Driver) - Load from .env or prompt
# ============================================================

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://172.31.240.1:7691")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD") or getpass("Enter Neo4j password: ")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"Neo4j connected: {NEO4J_URI}")

Neo4j connected: bolt://172.31.240.1:7691


### Step 2. Load Data
- Read product metadata and review corpus from `./data`

In [5]:
# =======================
# Data Loading Paths
# =======================
BASE_PATH = "./data" 
META_FILE = os.path.join(BASE_PATH, "amazon_face_core_meta.csv")
REVIEW_FILE = os.path.join(BASE_PATH, "amazon_face_core_reviews.csv")

### Step 3. Build Neo4j Graph
- Create nodes/relationships needed for:
  - Users
  - Products (with brand)
  - Purchase / interaction history

In [6]:
# Export Utility for result sharing

REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def _json_fallback(o):
    if is_dataclass(o):
        return asdict(o)
    if hasattr(o, "model_dump"):  # pydantic v2
        return o.model_dump()
    if hasattr(o, "dict"):        # pydantic v1
        return o.dict()
    if hasattr(o, "__dict__"):
        return o.__dict__
    return str(o)

def save_text(filename: str, text: str):
    p = REPORT_DIR / filename
    p.write_text(text, encoding="utf-8")
    print(f"[Saved] {p}")

def save_md(filename: str, md: str):
    save_text(filename, md)

def save_json(filename: str, obj):
    p = REPORT_DIR / filename
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_json_fallback), encoding="utf-8")
    print(f"[Saved] {p}")

In [7]:
# ==============================================
# Helper Functions: import products and reviews
# ==============================================

def init_neo4j_schema(driver):
    """
    Initialize database constraints to ensure data integrity and performance.
    Creates unique constraints on User ID, Product ID, Brand Name, etc.
    """
    print("Initializing Database Schema & Constraints...")
    queries = [
        "CREATE CONSTRAINT user_id IF NOT EXISTS FOR (u:User) REQUIRE u.user_id IS UNIQUE",
        "CREATE CONSTRAINT product_id IF NOT EXISTS FOR (p:Product) REQUIRE p.asin IS UNIQUE",
        "CREATE CONSTRAINT review_id IF NOT EXISTS FOR (r:Review) REQUIRE r.id IS UNIQUE",
        "CREATE CONSTRAINT brand_name IF NOT EXISTS FOR (b:Brand) REQUIRE b.name IS UNIQUE",
        "CREATE CONSTRAINT category_name IF NOT EXISTS FOR (c:Category) REQUIRE c.name IS UNIQUE"
    ]
    
    try:
        with driver.session() as session:
            for q in queries:
                session.run(q)
        print("Schema Constraints Created Successfully.")
    except Exception as e:
        print(f"Schema Initialization Warning: {e}")

def import_products(driver):
    """
    Import Product, Brand, and Category nodes from metadata CSV.
    """
    if not os.path.exists(META_FILE):
        print(f"Error: Meta file not found: {META_FILE}")
        return

    print(f"\nImporting Products from: {META_FILE}")
    
    df_meta = pd.read_csv(META_FILE)
    df_meta.fillna({'brand': 'Unknown', 'title': 'No Title', 'price': 0}, inplace=True)
    
    print(f"   - Total Products to Import: {len(df_meta):,}")
    
    # Cypher Query: Create Product, Brand, Category and their relationships
    query = """
    UNWIND $rows AS row
    MERGE (p:Product {asin: row.parent_asin})
    SET p.title = row.title, 
        p.price = toFloat(row.price), 
        p.features = row.features
        
    MERGE (b:Brand {name: row.brand})
    MERGE (p)-[:MADE_BY]->(b)
    
    MERGE (c:Category {name: 'Face'})
    MERGE (p)-[:BELONGS_TO]->(c)
    """
    
    # Batch Processing
    batch_size = 1000
    with driver.session() as session:
        for i in tqdm(range(0, len(df_meta), batch_size), desc="Processing Batches"):
            batch = df_meta.iloc[i:i+batch_size].to_dict('records')
            session.run(query, rows=batch)
            
    print("Product Import Completed.")

def import_reviews(driver):
    """
    Import User and Review nodes, creating relationships between Users, Reviews, and Products.
    """
    if not os.path.exists(REVIEW_FILE):
        print(f"Error: Review file not found: {REVIEW_FILE}")
        return

    print(f"\nImporting Reviews from: {REVIEW_FILE}")
    df_review = pd.read_csv(REVIEW_FILE)
    
    df_review['review_id'] = df_review['user_id'] + '_' + df_review['parent_asin']
    
    print(f"   - Total Reviews to Import: {len(df_review):,}")
    
    # Cypher Query: Create User, Review and link them
    query = """
    UNWIND $rows AS row
    
    MERGE (u:User {user_id: row.user_id})
    
    MERGE (r:Review {id: row.review_id})
    SET r.rating = toInteger(row.rating), 
        r.text = row.text, 
        r.timestamp = row.timestamp, 
        r.helpful_vote = toInteger(row.helpful_vote)
        
    MERGE (u)-[:WROTE]->(r)
    
    WITH r, row
    MATCH (p:Product {asin: row.parent_asin})
    MERGE (r)-[:EVALUATED]->(p)
    """
    
    batch_size = 1000
    with driver.session() as session:
        for i in tqdm(range(0, len(df_review), batch_size), desc="Processing Batches"):
            batch = df_review.iloc[i:i+batch_size].to_dict('records')
            session.run(query, rows=batch)
            
    print("Review Import Completed.")

In [8]:
# ==========================================
# Execution
# ==========================================
init_neo4j_schema(driver)
import_products(driver)
import_reviews(driver)

print("✔ All Neo4j imports completed successfully.")

Initializing Database Schema & Constraints...
Schema Constraints Created Successfully.

Importing Products from: ./data/amazon_face_core_meta.csv
   - Total Products to Import: 20,082


Processing Batches: 100%|███████████████████████████████████████████████████████████████| 21/21 [00:02<00:00,  8.91it/s]


Product Import Completed.

Importing Reviews from: ./data/amazon_face_core_reviews.csv
   - Total Reviews to Import: 115,250


Processing Batches: 100%|█████████████████████████████████████████████████████████████| 116/116 [00:08<00:00, 13.86it/s]

Review Import Completed.
✔ All Neo4j imports completed successfully.


### Step 4. Vector Indexing & Embeddings
- Generate embeddings for reviews
- Build Neo4j vector index (`review_embedding_index`)

In [9]:
# ==========================================
# Vector_indexing: Configuration
# ==========================================

EMBEDDING_DIMENSION = 1536
MAX_EMBEDDING_CHARS = 2000
BATCH_SIZE = 1000  # Adjust based on API rate limits

In [10]:
# ==========================================
# Helper Functions
# ==========================================

def get_reviews_without_embeddings(driver, limit=1000):
    """
    Fetch reviews that do not have an embedding property yet.
    """
    query = """
    MATCH (r:Review)
    WHERE r.embedding IS NULL AND r.text IS NOT NULL
    RETURN r.id AS id, r.text AS text
    LIMIT $limit
    """
    with driver.session() as session:
        result = session.run(query, limit=limit)
        return [{"id": record["id"], "text": record["text"]} for record in result]

def update_review_embeddings(driver, updates):
    """
    Update Review nodes using standard SET query.
    """
    query = """
    UNWIND $updates AS row
    MATCH (r:Review {id: row.id})
    SET r.embedding = row.embedding
    """
    try:
        with driver.session() as session:
            session.run(query, updates=updates)
    except Exception as e:
        print(f"Critical Error updating embeddings: {e}")
        raise e

def create_vector_index(driver):
    """
    Create Vector Index if it doesn't exist.
    """
    index_name = "review_embedding_index"
    print(f"Creating/Verifying Vector Index: {index_name}...")
    
    check_query = "SHOW INDEXES WHERE name = $name"
    with driver.session() as session:
        result = session.run(check_query, name=index_name)
        if result.peek():
            print(f"✔ Index '{index_name}' already exists. Skipping creation.")
            return

    create_query = f"""
    CREATE VECTOR INDEX {index_name} IF NOT EXISTS
    FOR (r:Review)
    ON (r.embedding)
    OPTIONS {{indexConfig: {{
      `vector.dimensions`: 1536,
      `vector.similarity_function`: 'cosine'
    }}}}
    """
    try:
        with driver.session() as session:
            session.run(create_query)
        print(f"✔ Successfully created vector index: {index_name}")
        print("Waiting for index to be online...")
        time.sleep(5)
    except Exception as e:
        print(f"⚠ Failed to create index: {e}")

def generate_embeddings(text_list):
    """
    Generate embeddings using the global OpenAI client.
    """
    try:
        clean_texts = []
        for text in text_list:
            if not isinstance(text, str):
                text = str(text)
            text = text.replace("\n", " ")
            text = text[:MAX_EMBEDDING_CHARS] 
            clean_texts.append(text)

        # Use global 'client' and 'EMBEDDING_MODEL'
        response = client.embeddings.create(
            input=clean_texts,
            model=EMBEDDING_MODEL
        )
        return [data.embedding for data in response.data]

    except Exception as e:
        print(f"OpenAI API Error: {e}")
        return []

In [11]:
# ==========================================
# Execution: Embedding & Indexing
# ==========================================
try:
    print("\nStarting Full Vector Embedding Generation...")
    total_processed = 0
    
    # Use global 'driver' directly
    while True:
        # Fetch batch
        batch = get_reviews_without_embeddings(driver, limit=BATCH_SIZE)
        
        if not batch:
            print("✔ No more reviews to process. All embeddings generated.")
            break
        
        texts = [item["text"] for item in batch]
        ids = [item["id"] for item in batch]
        
        print(f"Generating embeddings for batch of {len(texts)} reviews...")
        start_time = time.time()
        
        # API Call (using global client inside function)
        embeddings = generate_embeddings(texts)
        
        if not embeddings:
            print("Failed to generate embeddings. Stopping process.")
            break
            
        # Update DB
        update_data = [{"id": uid, "embedding": emb} for uid, emb in zip(ids, embeddings)]
        update_review_embeddings(driver, update_data)
        
        total_processed += len(batch)
        elapsed = time.time() - start_time
        print(f"   - Processed {total_processed} reviews successfully. (Batch time: {elapsed:.2f}s)")

    # Create Index after processing
    print("\nBuilding Vector Index...")
    create_vector_index(driver)
    
    print("\n✔ All Embedding Steps Completed Successfully!")

except Exception as e:
    print(f"\nError: {e}")


Starting Full Vector Embedding Generation...
✔ No more reviews to process. All embeddings generated.

Building Vector Index...
Creating/Verifying Vector Index: review_embedding_index...
✔ Index 'review_embedding_index' already exists. Skipping creation.

✔ All Embedding Steps Completed Successfully!


### Step 5. Hybrid Retrieval
- Core algorithm implemented to fuse semantic similarity with domain-specific graph signals.
- run_hybrid_query defined to combine vector scores with log-weighted brand loyalty scores.
- generate_answer implemented to synthesize personalized recommendations using the LLM.

In [12]:
# ====================================================================
# Helper Functions: Dynamic Brand Loyalty Weighting & Hybrid Retrieval
# ====================================================================

def get_embedding(text):
    """
    Generate embedding vector using Global Client.
    """
    clean = str(text).replace("\n", " ")
    # Use global 'client' and 'EMBEDDING_MODEL'
    return client.embeddings.create(
        input=[clean], 
        model=EMBEDDING_MODEL
    ).data[0].embedding
    
def analyze_user_loyalty(driver, user_id):
    """
    Analyze user's brand loyalty based on purchase history.
    Returns: loyalty_score (0.0~1.0), top_brand_name, product_list
    """
    query = """
    MATCH (u:User {user_id: $user_id})-[:WROTE]->(:Review)-[:EVALUATED]->(p:Product)-[:MADE_BY]->(b:Brand)
    WITH u, p, b
    WITH count(p) AS total_count, 
         b.name AS brand_name, 
         count(b) AS brand_count,
         collect(p.title) AS products
    ORDER BY brand_count DESC
    LIMIT 1
    RETURN total_count, brand_name, brand_count, products
    """
    
    with driver.session() as session:
        result = session.run(query, user_id=user_id).single()
        
        if result:
            total = result['total_count']
            # If purchase history is too small (<3), treat as low loyalty
            if total < 3:
                return 0.1, result['brand_name'], result['products']
            
            score = result['brand_count'] / total
            return score, result['brand_name'], result['products']
            
    return 0.0, None, []

In [13]:
# Dynamic Hybrid Retrieval

def run_hybrid_query(driver, query_embedding, user_id=None, loyalty_score=0.0, k=5, exclude_purchased=False, return_context_string=False):
    """
    Unified Hybrid Search Function.
    - Combines Vector Search + Graph Brand History Boosting.
    - Replaces multiple redundant functions in previous cells.
    
    Args:
        return_context_string (bool): If True, returns formatted strings for RAGAS. If False, returns data dicts.
    """
    # 1. Calculate Dynamic Weight
    brand_weight = loyalty_score * 0.5 
    
    # 2. Construct Cypher Query
    # We inject the WHERE clause dynamically using f-string because Cypher doesn't support conditional blocks well.
    # But we use parameters ($embedding, $weight) for values to ensure safety.
    
    filter_clause = ""
    if exclude_purchased and user_id:
        filter_clause = "WHERE NOT EXISTS { MATCH (u:User {user_id: $user_id})-[:WROTE]->(:Review)-[:EVALUATED]->(product) }"

    cypher_query = f"""
    // 1. Vector Search
    // Leveraging Neo4j's native ANN index to eliminate external DB overhead and minimize latency.
    CALL db.index.vector.queryNodes('review_embedding_index', 150, $embedding)
    YIELD node AS similar_review, score AS vector_score

    // 2. Graph Traversal
    MATCH (similar_review)-[:EVALUATED]->(product:Product)-[:MADE_BY]->(brand:Brand)
    
    // Optional Filter (Exclude Purchased)
    {filter_clause}

    // 3. Personalization (Check History)
    OPTIONAL MATCH (u:User {{user_id: $user_id}})-[:WROTE]->(:Review)-[:EVALUATED]->(:Product)-[:MADE_BY]->(brand)
    WITH product, brand, similar_review, vector_score, count(u) AS history_count

    // 4. Re-ranking Logic (Dynamic Weight Applied via Parameter)
    WITH product, brand, vector_score, history_count, similar_review,
         (vector_score * (1 + (log(1 + history_count) * $brand_weight))) AS final_score

    ORDER BY final_score DESC
    LIMIT $k
    
    RETURN product.title AS product_name,
           brand.name AS brand_name,
           product.features AS features,
           similar_review.text AS review_text,
           vector_score,
           history_count,
           final_score
    """
    
    with driver.session() as session:
        result = session.run(cypher_query, embedding=query_embedding, user_id=user_id, brand_weight=brand_weight, k=k)
        records = [record.data() for record in result]

    # Ensure JSON-serializable outputs (defensive)
    records = [
        r if isinstance(r, dict) else (getattr(r, "__dict__", {"value": str(r)}))
        for r in records
    ]

    # Return Format 1: String List for RAGAS
    if return_context_string:
        return [f"{r['product_name']} by {r['brand_name']}: {r['review_text']}" for r in records]

    # Return Format 2: Dictionary List for Engine/Demo
    return records

In [14]:
def generate_answer(client, query, context, user_id=None, top_brand=None):
    """
    Generate answer using LLM based on retrieved context.
    """
    system_prompt = """
    You are an expert Personal Shopper AI.
    Recommend products based on the user's query and the retrieved candidates.
    
    Analysis of User Style:
    - If the user has a 'Favorite Brand', acknowledge it but also introduce high-quality alternatives.
    - Explain clearly why each product fits their specific need (based on features/reviews).
    """
    
    context_text = ""
    for i, item in enumerate(context):
        # Handle case where context might be simple strings (RAGAS) or dicts (Engine)
        if isinstance(item, str):
            context_text += f"\n[Candidate #{i+1}] {item}"
        else:
            context_text += f"""
            [Candidate #{i+1}]
            - Product: {item.get('product_name')}
            - Brand: {item.get('brand_name')}
            - Relevance Score: {item.get('final_score'):.4f}
            - User's Brand History: {item.get('history_count')} times purchased
            - Key Review Snippet: "{item.get('review_text', '')[:100]}..."
            """
    
    user_prompt = f"""
    User Query: {query}
    User ID: {user_id}
    User's Favorite Brand: {top_brand if top_brand else "None"}
    
    [Market Data / Context]:
    {context_text}
    """

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content


### Step 6. Case Study (Dynamic Personalization)
- Same query, different user IDs → different rankings via graph signal

In [15]:
# ==========================================
# Case Study: Qualitative Evaluation
# ==========================================

# Target User ID provided by you
TARGET_USER_ID = "AEXCLMGS3Y7SRW5CMLJBNYI2HBZQ"

def get_user_last_review_text(tx, user_id):
    """
    Fetch the last review written by the user to use as a query.
    """
    query = """
    MATCH (u:User {user_id: $user_id})-[:WROTE]->(r:Review)-[:EVALUATED]->(p:Product)
    RETURN r.text AS text, p.title AS product
    ORDER BY r.timestamp DESC
    LIMIT 1
    """
    result = tx.run(query, user_id=user_id).single()
    return (result['text'], result['product']) if result else (None, None)

In [16]:
# ==========================================
# Execution: Recommendation Demo
# ==========================================

TOP_K = 10
LOYALTY_SCORE = 0.2
EXCLUDE_PURCHASED = False

try:
    with driver.session() as session:
        # Step 1: Fetch last review text as query + ground truth product
        review_text, last_product = session.execute_read(get_user_last_review_text, TARGET_USER_ID)

    if not review_text:
        print("User has no reviews found.")
    else:
        print(f"[Context] Last Purchase: {last_product[:60]}...")
        print(f"[Query] Review Text: \"{review_text[:120]}...\"")

        # Step 2: Embed
        query_vec = get_embedding(review_text)

        # Step 3: Hybrid retrieval
        print(f"\nExecuting Hybrid Query for User: {TARGET_USER_ID}")
        rec_results = run_hybrid_query(
            driver,
            query_embedding=query_vec,
            user_id=TARGET_USER_ID,
            loyalty_score=LOYALTY_SCORE,
            k=TOP_K,
            exclude_purchased=EXCLUDE_PURCHASED
        )

        print(f"\n***[Ground Truth] User Actually Bought: {last_product}")

        print("\n[Recommendation Results]")
        print(f"{'Brand':<20} | {'History':<7} | {'VecScore':<9} | {'FinalScore':<10} | Product")
        print("-" * 105)

        for r in rec_results:
            history = int(r.get("history_count", 0) or 0)
            mark = "*" if history > 0 else " "
            brand = str(r.get("brand_name", ""))[:18]
            vec = float(r.get("vector_score", 0.0) or 0.0)
            fin = float(r.get("final_score", 0.0) or 0.0)
            prod = str(r.get("product_name", ""))[:55]
            print(f"{mark} {brand:<18} | {history:<7} | {vec:<9.4f} | {fin:<10.4f} | {prod}...")

except Exception as e:
    print(f"\nError: {e}")

[Context] Last Purchase: First Aid Beauty Facial Radiance Pads – Daily Exfoliating Pa...
[Query] Review Text: "It soothing.love the way.my skin feels..."


Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {}, '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    // 1. Vector Search\n    // Leveraging Neo4j's native ANN index to eliminate external DB overhead and minimize latency.\n    CALL db.index.vector.queryNodes('review_embedding_index', 150, $embedding)\n    YIELD node AS similar_review, score AS vector_score\n\n    // 2. Graph Traversal\n    MATCH (similar_review)-[:EVALUATED]->(product:Product)-[:MADE_BY]->(brand:Brand)\n\n    // Optional Filter (Exclude Purchased)\n    \n\n    // 3. Personalization (Check History)\n 


Executing Hybrid Query for User: AEXCLMGS3Y7SRW5CMLJBNYI2HBZQ

***[Ground Truth] User Actually Bought: First Aid Beauty Facial Radiance Pads – Daily Exfoliating Pads with AHA that Help Tone & Brighten Skin – 28 Count

[Recommendation Results]
Brand                | History | VecScore  | FinalScore | Product
---------------------------------------------------------------------------------------------------------
* First Aid Beauty   | 1       | 0.9997    | 1.0690     | First Aid Beauty Facial Radiance Pads – Daily Exfoliati...
* TruSkin Naturals   | 1       | 0.8613    | 0.9210     | TruSkin Vitamin C Serum for Face – Anti Aging Face Seru...
  Juvenu             | 0       | 0.9068    | 0.9068     | Juvenu Skin Intensive Serum Probiotic serum | Help trea...
* First Aid Beauty   | 1       | 0.8448    | 0.9034     | First Aid Beauty Facial Radiance Pads – Daily Exfoliati...
  Palmer's           | 0       | 0.9002    | 0.9002     | Palmer's Cocoa Butter Formula Moisturizing Skin Therapy...

### Step 7. Evaluation: Quantitative Performance Metrics
* Hit Rate@10 and NDCG@10 measured on a test set of 50 users:
    * **Hit Rate@10:** Measures whether the target product appears within the top-10 results. (Formula: $1$ if $Target \in Top10$ else $0$)
    * **NDCG@10:** Evaluates ranking quality by prioritizing correct items at higher positions. (Formula: $1 / \log_2(Rank + 1)$)

In [17]:
# ==========================================
# Test Settings
# ==========================================

SAMPLE_SIZE = 50  
TOP_K = 10        # Top-10 recommendations

In [18]:
# ==========================================
# Helper Functions: Evaluation
# ==========================================

def fetch_test_dataset(driver, sample_size=50):
    """
    Fetch 'Core Users' and their 'Last Review' to create a Ground Truth dataset.
    """
    print(f"Building Test Dataset (Sample: {sample_size})...")
    query = """
    MATCH (u:User)-[:WROTE]->(r:Review)-[:EVALUATED]->(p:Product)
    WITH u, count(r) AS review_count
    WHERE review_count >= 5  // Only Core Users
    WITH u
    ORDER BY rand() LIMIT $limit
    
    MATCH (u)-[:WROTE]->(r:Review)-[:EVALUATED]->(target_p:Product)-[:MADE_BY]->(target_b:Brand)
    WITH u, target_p, target_b, r
    ORDER BY r.timestamp DESC
    WITH u, head(collect(target_p.title)) AS target_product, 
            head(collect(r.text)) AS raw_review_text,
            head(collect(target_b.name)) AS target_brand
            
    MATCH (u)-[:WROTE]->(:Review)-[:EVALUATED]->(p:Product)-[:MADE_BY]->(b:Brand)
    WITH u, target_product, raw_review_text, target_brand, count(p) as total, b, count(b) as b_count
    ORDER BY b_count DESC
    WITH u, target_product, raw_review_text, target_brand, total, head(collect(b.name)) as top_brand, head(collect(b_count)) as top_count
    
    RETURN u.user_id as user_id, 
           target_product, 
           raw_review_text, 
           (toFloat(top_count)/total) as loyalty_score
    """
    with driver.session() as session:
        result = session.run(query, limit=sample_size)
        return [record.data() for record in result]

def generate_synthetic_query(review_text):
    """
    [NEW] Convert a specific product review into a generic user search query.
    This prevents 'Data Leakage' where the review text exactly matches the target document.
    """
    prompt = f"""
    Task: You are a user looking for a skincare product. 
    Based on the review you eventually wrote (below), reconstruct the **Search Query** or **Need Statement** you likely had BEFORE buying the product.
    
    Review: "{review_text[:400]}..."
    
    Rules:
    1. Do NOT mention the specific product name or brand.
    2. Focus on skin concerns (e.g., dry, acne) and desired features (e.g., hydrating, scent-free).
    3. Keep it natural and short (10-15 words).
    4. Example: "I need a gentle toner for my sensitive skin that helps with redness."
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        return response.choices[0].message.content.strip().replace('"', '')
    except:
        return review_text # Fallback to original if API fails

def run_search(driver, embedding, user_id, loyalty_score, mode='hybrid'):
    """
    Execute search wrapper for evaluation.
    """
    # Determine effective loyalty score
    effective_loyalty = loyalty_score if mode == 'hybrid' else 0.0

    # Reuse the unified search function
    results = run_hybrid_query(
        driver,
        query_embedding=embedding,
        user_id=user_id,
        loyalty_score=effective_loyalty,
        k=TOP_K,
        exclude_purchased=False, 
        return_context_string=False
    )
    return [r['product_name'] for r in results]

def calculate_metrics(recommendations, target_product):
    """Calculate Hit Rate and NDCG."""
    try:
        rank = recommendations.index(target_product) + 1
        hit = 1.0
        ndcg = 1.0 / np.log2(rank + 1)
    except ValueError:
        hit = 0.0
        ndcg = 0.0
    return hit, ndcg

In [19]:
# ==========================================
# Quantitative Evaluation: Execution Loop
# ==========================================

print(f"Connecting to Neo4j at {NEO4J_URI} (Using Global Driver)...")
driver.verify_connectivity()

# Prepare Test Data
test_data = fetch_test_dataset(driver, sample_size=SAMPLE_SIZE)
print(f"Fetched {len(test_data)} raw test cases.")

# Pre-processing: Generate Synthetic Queries
print("\n Generating Synthetic Queries (Paraphrasing) to reduce data leakage...")
print("   (Converting specific reviews into general user needs...)")

for case in tqdm(test_data, desc="Paraphrasing"):
    # Convert raw review -> Generic Search Query
    case['query_text'] = generate_synthetic_query(case['raw_review_text'])

# Show an example of the transformation
print(f"\n[Example Transformation]")
print(f"Original Review: {test_data[0]['raw_review_text'][:80]}...")
print(f"Generated Query: {test_data[0]['query_text']}")
print("-" * 50)

quant_scores = {
    'vector': {'hit': [], 'ndcg': []},
    'hybrid': {'hit': [], 'ndcg': []}
}

print("\nStarting Evaluation (Comparing Vector vs. Hybrid)...")

# Evaluation Loop
try:
    for case in tqdm(test_data, desc="Evaluating"):
        user_id = case['user_id']
        target = case['target_product']
        loyalty = case['loyalty_score']
        query_text = case['query_text'] # Uses the NEW synthetic query

        # Use global client inside get_embedding
        query_vec = get_embedding(query_text)

        # Vector Only Search
        recs_vector = run_search(driver, query_vec, user_id, loyalty, mode='vector')
        h_v, n_v = calculate_metrics(recs_vector, target)
        quant_scores['vector']['hit'].append(h_v)
        quant_scores['vector']['ndcg'].append(n_v)

        # Hybrid Search
        recs_hybrid = run_search(driver, query_vec, user_id, loyalty, mode='hybrid')
        h_h, n_h = calculate_metrics(recs_hybrid, target)
        quant_scores['hybrid']['hit'].append(h_h)
        quant_scores['hybrid']['ndcg'].append(n_h)
        
    print("\n✔ Evaluation Loop Completed.")

except Exception as e:
    print(f"\nError during evaluation loop: {e}")

Connecting to Neo4j at bolt://172.31.240.1:7691 (Using Global Driver)...
Building Test Dataset (Sample: 50)...
Fetched 93 raw test cases.

 Generating Synthetic Queries (Paraphrasing) to reduce data leakage...
   (Converting specific reviews into general user needs...)


Paraphrasing: 100%|█████████████████████████████████████████████████████████████████████| 93/93 [01:18<00:00,  1.18it/s]



[Example Transformation]
Original Review: A good lightweight gel moisturizer. It's a cool blue color and doesn't have a st...
Generated Query: Looking for a lightweight gel moisturizer that soothes and hydrates without strong scent.
--------------------------------------------------

Starting Evaluation (Comparing Vector vs. Hybrid)...


Evaluating:   0%|                                                                                | 0/93 [00:00<?, ?it/s]Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {}, '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    // 1. Vector Search\n    // Leveraging Neo4j's native ANN index to eliminate external DB overhead and minimize latency.\n    CALL db.index.vector.queryNodes('review_embedding_index', 150, $embedding)\n    YIELD node AS similar_review, score AS vector_score\n\n    // 2. Graph Traversal\n    MATCH (similar_review)-[:EVALUATED]->(product:Product)-[:MAD


✔ Evaluation Loop Completed.


In [20]:
# ==========================================
# Quantitative Evaluation: Report
# ==========================================

# Calculate Averages
avg_v_hit = np.mean(quant_scores['vector']['hit']) * 100
avg_v_ndcg = np.mean(quant_scores['vector']['ndcg']) * 100
avg_h_hit = np.mean(quant_scores['hybrid']['hit']) * 100
avg_h_ndcg = np.mean(quant_scores['hybrid']['ndcg']) * 100

print("\n" + "="*40)
print("FINAL EVALUATION REPORT")
print("="*40)

# Print Table
print(f"{'Metric':<12} | {'Vector Only':<11} | {'GraphRAG (Hybrid)':<17} | {'Improvement'}")
print(f"{'-'*12}|{'-'*13}|{'-'*19}|{'-'*12}")
print(f"{'Hit Rate@10':<12} | {avg_v_hit:6.2f}%     | {avg_h_hit:6.2f}%            | {avg_h_hit - avg_v_hit:+.2f}%p")
print(f"{'NDCG@10':<12} | {avg_v_ndcg:6.2f}%     | {avg_h_ndcg:6.2f}%            | {avg_h_ndcg - avg_v_ndcg:+.2f}%p")
print("="*40)

# Conclusion
if avg_h_hit > avg_v_hit:
    print("Conclusion: GraphRAG outperforms Vector Search.")
else:
    print("Conclusion: Performance is similar. Check brand loyalty distribution.")


FINAL EVALUATION REPORT
Metric       | Vector Only | GraphRAG (Hybrid) | Improvement
------------|-------------|-------------------|------------
Hit Rate@10  |  21.51%     |  41.94%            | +20.43%p
NDCG@10      |  14.64%     |  30.92%            | +16.28%p
Conclusion: GraphRAG outperforms Vector Search.


In [21]:
quant_export = {
    "sample_size": SAMPLE_SIZE,
    "top_k": 10,
    "avg_vector_hit_rate_at_10": float(avg_v_hit),
    "avg_vector_ndcg_at_10": float(avg_v_ndcg),
    "avg_hybrid_hit_rate_at_10": float(avg_h_hit),
    "avg_hybrid_ndcg_at_10": float(avg_h_ndcg),
    "improvement_hit_rate_pp": float(avg_h_hit - avg_v_hit),
    "improvement_ndcg_pp": float(avg_h_ndcg - avg_v_ndcg),
}

save_json("eval_quantitative.json", quant_export)

[Saved] reports/eval_quantitative.json


### Step 7. Evaluation: Qualitative RAGAS
* Quality and factual consistency of AI responses assessed via the Ragas framework (Judge: GPT-4o).
* **High Answer Relevancy (0.85)** achieved, confirming the model's ability to generate advice aligned with specific user needs.
* **Faithfulness (0.66)** observed, reflecting a balance between retrieved context and the model's persuasive generation capabilities.
* **Context Precision (0.24):** Measures if the ground truth is ranked highly in the retrieved context. (Formula: $S_{relevant} / Total_{retrieved}$)

In [22]:
# ==========================================
# Qualitative Evaluation: RAGAS Settings
# ==========================================

SAMPLE_SIZE_RAGAS = 50
RAGAS_TOP_K = 3  # Contexts to retrieve

In [23]:
# ==========================================
# Helper Functions
# ==========================================

def generate_ragas_answer(query, contexts):
    """
    Generate an answer using the Service Model (gpt-4o-mini).
    """
    context_block = "\n".join([f"- {c}" for c in contexts])
    prompt = f"""
    User Query: {query}
    
    Contexts:
    {context_block}
    
    Answer the user query based on the contexts provided. Recommend the best product.
    """
    # Use Global Client (gpt-4o-mini defined in Step 1)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

In [24]:
# ==========================================
# Execution: Data Generation & Judging (RAGAS)
# ==========================================

import traceback

if not RUN_RAGAS:
    print("[Skip] RUN_RAGAS=False")
else:
    try:
        required = ["driver", "llm_judge", "emb"]
        missing = [x for x in required if x not in globals()]
        if missing:
            raise NameError(f"Missing required objects: {missing}. Run the setup cells first.")

        print(f"Connecting to Neo4j at {NEO4J_URI} (Using Global Driver)...")
        driver.verify_connectivity()

        test_cases = fetch_test_dataset(driver, sample_size=SAMPLE_SIZE_RAGAS)
        print(f"Prepared {len(test_cases)} raw test cases.")

        print("\nGenerating Synthetic Queries for RAGAS...")
        for case in tqdm(test_cases, desc="Paraphrasing"):
            case["query_text"] = generate_synthetic_query(case["raw_review_text"])

        # NOTE: This RAGAS version expects `reference` as a STRING (not a list).
        data_samples = {
            "question": [],
            "answer": [],
            "contexts": [],
            "reference": [],
        }

        print("\nGenerating Answers and Contexts...")

        for case in tqdm(test_cases, desc="Generating"):
            query_text = case.get("query_text", "")
            if not query_text:
                continue

            query_vec = get_embedding(query_text)

            contexts = run_hybrid_query(
                driver,
                query_embedding=query_vec,
                user_id=case.get("user_id"),
                loyalty_score=case.get("loyalty_score", 0.0),
                k=RAGAS_TOP_K,
                exclude_purchased=False,
                return_context_string=True
            )

            if not contexts:
                continue

            answer = generate_ragas_answer(query_text, contexts)

            data_samples["question"].append(query_text)
            data_samples["answer"].append(answer)
            data_samples["contexts"].append(contexts)

            gt = case.get("target_product", "")
            data_samples["reference"].append(str(gt))

        if len(data_samples["question"]) == 0:
            raise ValueError("No RAGAS samples generated (data_samples is empty).")

        i = 0
        print("\n[Debug] Sample schema check")
        print(" - question type:", type(data_samples["question"][i]))
        print(" - answer type:", type(data_samples["answer"][i]))
        print(" - contexts type:", type(data_samples["contexts"][i]), "| len:", len(data_samples["contexts"][i]))
        print(" - reference type:", type(data_samples["reference"][i]), "| value:", data_samples["reference"][i])

        print("\nRunning RAGAS Evaluation (Judge: GPT-4o)...")

        dataset = Dataset.from_dict(data_samples)

        metrics = [
            faithfulness,
            answer_relevancy,
            context_precision,
        ]

        ragas_result = evaluate(
            dataset,
            metrics=metrics,
            llm=llm_judge,
            embeddings=emb
        )

        print("✔ RAGAS Evaluation Completed.")

    except Exception as e:
        print("\nError during RAGAS evaluation:", repr(e))
        traceback.print_exc()

print("[System] ragas_result exists:", "ragas_result" in globals())

Connecting to Neo4j at bolt://172.31.240.1:7691 (Using Global Driver)...
Building Test Dataset (Sample: 50)...
Prepared 84 raw test cases.

Generating Synthetic Queries for RAGAS...


Paraphrasing: 100%|█████████████████████████████████████████████████████████████████████| 84/84 [01:06<00:00,  1.25it/s]



Generating Answers and Contexts...


Generating: 100%|███████████████████████████████████████████████████████████████████████| 84/84 [04:13<00:00,  3.02s/it]



[Debug] Sample schema check
 - question type: <class 'str'>
 - answer type: <class 'str'>
 - contexts type: <class 'list'> | len: 3
 - reference type: <class 'str'> | value: ANAI RUI Turmeric Facial Mask, Vitamin C Clay Mask with Wood Brush, Skin Care Mask for Acne, Refining Pores, Radiant & Smooth Skin, 6.35 oz

Running RAGAS Evaluation (Judge: GPT-4o)...


Evaluating:   0%|          | 0/252 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

✔ RAGAS Evaluation Completed.
[System] ragas_result exists: True


In [25]:
# ==========================================
# Qualitative Evaluation: Report
# ==========================================

if not RUN_RAGAS:
    print("[Skip] RUN_RAGAS=False (RAGAS report is disabled)")
elif "ragas_result" not in globals():
    print("[Skip] ragas_result is not defined. Run the RAGAS execution cell first.")
else:
    print("=" * 40)
    print("RAGAS EVALUATION REPORT")
    print("=" * 40)

    df_results = ragas_result.to_pandas()

    all_cols = df_results.columns.tolist()
    print(f"[System] Detected Columns: {all_cols}")

    def get_existing_col(candidates, columns):
        for col in candidates:
            if col in columns:
                return col
        return None

    col_question = get_existing_col(["user_input", "question"], all_cols)
    col_answer = get_existing_col(["response", "answer"], all_cols)

    target_metrics = ["faithfulness", "answer_relevancy", "context_precision"]
    existing_metrics = [m for m in target_metrics if m in all_cols]

    total_rows = len(df_results)
    cleanup_subset = [m for m in ["faithfulness", "answer_relevancy"] if m in existing_metrics]
    df_clean = df_results.dropna(subset=cleanup_subset) if cleanup_subset else df_results

    clean_count = len(df_clean)
    missing_count = total_rows - clean_count

    print(f"\n[Data Status]")
    print(f"- Total Sample Count: {total_rows}")
    print(f"- Analyzable Sample Count: {clean_count} (Success Rate: {clean_count/total_rows*100:.1f}%)")
    print(f"- Missing/Failed Sample Count: {missing_count}")

    print("-" * 40)
    print("[Final Performance Evaluation Results (Mean)]")

    if clean_count > 0 and existing_metrics:
        for metric in existing_metrics:
            print(f" - {metric}: {df_clean[metric].mean():.4f}")
    else:
        print(" No valid data available to calculate means.")

    print("\n[Top 3 Samples based on Answer Relevancy]")

    display_cols = []
    if col_question: display_cols.append(col_question)
    if col_answer: display_cols.append(col_answer)
    display_cols.extend(existing_metrics)

    if "answer_relevancy" in df_clean.columns and not df_clean.empty:
        print(df_clean[display_cols].sort_values(by="answer_relevancy", ascending=False).head(3))
    elif not df_clean.empty:
        print(df_clean[display_cols].head(3))
    else:
        print("No data to display.")

RAGAS EVALUATION REPORT
[System] Detected Columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision']

[Data Status]
- Total Sample Count: 84
- Analyzable Sample Count: 84 (Success Rate: 100.0%)
- Missing/Failed Sample Count: 0
----------------------------------------
[Final Performance Evaluation Results (Mean)]
 - faithfulness: 0.7619
 - answer_relevancy: 0.8187
 - context_precision: 0.2560

[Top 3 Samples based on Answer Relevancy]
                                           user_input  \
37  Looking for a creamy retinol product to reduce...   
30  I need a strong acne treatment that effectivel...   
81  I need a foaming cleanser that removes makeup ...   

                                             response  faithfulness  \
37  Based on your request for a creamy retinol pro...      0.700000   
30  For a strong acne treatment that effectively d...      0.800000   
81  Based on your need for a foaming cleanser that.

In [26]:
# ==========================================
# Export: RAGAS results
# ==========================================

if not RUN_RAGAS:
    print("[Skip] RUN_RAGAS=False (RAGAS export is disabled)")
elif "ragas_result" not in globals():
    print("[Skip] ragas_result is not defined (run the RAGAS execution cell first)")
else:
    df_results = ragas_result.to_pandas()

    # Metrics to summarize
    target_metrics = ["faithfulness", "answer_relevancy", "context_precision"]
    existing_metrics = [m for m in target_metrics if m in df_results.columns]

    # Cleaning rule
    cleanup_subset = [m for m in ["faithfulness", "answer_relevancy"] if m in existing_metrics]
    df_clean = df_results.dropna(subset=cleanup_subset) if cleanup_subset else df_results

    # Build Top-3 table (Excel only)
    display_cols = []
    for c in ["user_input", "question"]:
        if c in df_clean.columns:
            display_cols.append(c)
            break
    for c in ["response", "answer"]:
        if c in df_clean.columns:
            display_cols.append(c)
            break
    if existing_metrics:
        display_cols.extend(existing_metrics)

    if "answer_relevancy" in df_clean.columns and not df_clean.empty and display_cols:
        df_top3 = df_clean[display_cols].sort_values(by="answer_relevancy", ascending=False).head(3)
    elif not df_clean.empty and display_cols:
        df_top3 = df_clean[display_cols].head(3)
    else:
        df_top3 = pd.DataFrame()

    # Export Excel (single workbook with sheets: raw/clean/top3)
    try:
        out_xlsx = REPORT_DIR / "ragas_export.xlsx"
        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            df_results.to_excel(writer, sheet_name="raw", index=False)
            df_clean.to_excel(writer, sheet_name="clean", index=False)
            df_top3.to_excel(writer, sheet_name="top3", index=False)
        print(f"[Saved] {out_xlsx}")
    except Exception as e:
        print(f"[Skip] Excel export failed: {e}")

    # Export clean TXT summary (NO Top-3 section)
    try:
        all_cols = df_results.columns.tolist()
        total_rows = len(df_results)
        clean_count = len(df_clean)
        missing_count = total_rows - clean_count

        report_lines = []
        report_lines.append("=" * 40)
        report_lines.append("RAGAS EVALUATION REPORT")
        report_lines.append("=" * 40)
        report_lines.append(f"[System] Detected Columns: {all_cols}")
        report_lines.append("")
        report_lines.append("[Data Status]")
        report_lines.append(f"- Total Sample Count: {total_rows}")
        report_lines.append(f"- Analyzable Sample Count: {clean_count} (Success Rate: {clean_count/total_rows*100:.1f}%)")
        report_lines.append(f"- Missing/Failed Sample Count: {missing_count}")
        report_lines.append("")
        report_lines.append("[Final Performance Evaluation Results (Mean)]")

        if clean_count > 0 and existing_metrics:
            for metric in existing_metrics:
                report_lines.append(f"- {metric}: {df_clean[metric].mean():.4f}")
        else:
            report_lines.append("No valid data available to calculate means.")

        out_txt = REPORT_DIR / "ragas_report.txt"
        out_txt.write_text("\n".join(report_lines), encoding="utf-8")
        print(f"[Saved] {out_txt}")
    except Exception as e:
        print(f"[Skip] TXT export failed: {e}")

[Saved] reports/ragas_export.xlsx
[Saved] reports/ragas_report.txt


## Step 8. Interactive Demo
* Interactive interface provided for real-time testing via manual input of User IDs and queries.
* RAG retrieval and LLM generation processes observed dynamically.
* System adaptability verified across diverse scenarios, including "Cold Start" cases and specific user contexts.

In [27]:
# Result export

demo_log_path = REPORT_DIR / "interactive_demo_log.jsonl"
print(f"[System] Demo logs will be saved to: {demo_log_path}")

[System] Demo logs will be saved to: reports/interactive_demo_log.jsonl


In [28]:
# ==========================================
# Execution: Interactive Demo
# ==========================================

# Ensure prerequisite steps are run
if 'client' not in locals() or 'driver' not in locals():
    raise ValueError(" Please run the previous 'Global Setup' and 'Helper Functions' cells first.")

print("\nDynamic GraphRAG Engine Initialized! (Ready for Inputs)")

try:
    # Use Global Driver (No need to reconnect)
    driver.verify_connectivity()
    
    while True:
        print("\n" + "="*60)
        # 1. User ID Input
        user_input = input("User ID (Press Enter for Cold Start, 'exit' to quit): ").strip()
        if user_input.lower() == 'exit': 
            print("Exiting Demo. Goodbye! ")
            break
            
        user_id = user_input if user_input else None
        
        # 2. Analyze User Profile
        loyalty_score, top_brand, purchased_list = 0.0, None, []
        if user_id:
            loyalty_score, top_brand, purchased_list = analyze_user_loyalty(driver, user_id)
            print(f"[User Profile] Loyalty Score: {loyalty_score:.2f} (Favorite: {top_brand})")
            print(f"   Already Purchased: {len(purchased_list)} items")

        # 3. Query Input
        query_input = input("Your Question: ").strip()
        if query_input.lower() == 'exit': 
            break
        if not query_input: 
            query_input = "Recommend something based on my taste."
        
        # 4. Set Search Mode
        mode = input("Mode? (1: Discover New, 2: Repurchase/Loyalty) [Default: 1]: ").strip()
        exclude_purchased = False if mode == '2' else True
        
        print(f"\nSearching... (Exclude Purchased: {exclude_purchased})")
        
        # 5. Embed & Search (Unified Function)
        # [FIX] Use global client inside get_embedding (remove 'client' arg)
        query_vec = get_embedding(query_input)
        
        # [FIX] Use 'run_hybrid_query' instead of 'hybrid_search_dynamic'
        retrieved_items = run_hybrid_query(
            driver,
            query_embedding=query_vec,
            user_id=user_id,
            loyalty_score=loyalty_score,
            exclude_purchased=exclude_purchased,
            k=RAGAS_TOP_K,
            return_context_string=False  # Return dicts for display
        )
        
        if not retrieved_items:
            print("No products found.")
            continue
            
        # 6. Show Results
        print(f"Top 5 Recommendations:")
        for item in retrieved_items:
            tag = "[Brand Pick]" if item['brand_name'] == top_brand else "✨ [New Discovery]"
            print(f"   - {tag} {item['product_name'][:40]}... (Score: {item['final_score']:.4f})")
        
        # 7. Generate AI Response
        print("\nGenerating Insight...")
        # Note: generate_answer still takes 'client' as per our previous definition
        answer = generate_answer(client, query_input, retrieved_items, user_id, top_brand)

        # Log the full interactive turn for reproducibility (inputs, recs, answer)
        turn_log = {
            "user_id": user_id,
            "loyalty_score": float(loyalty_score),
            "top_brand": top_brand,
            "exclude_purchased": bool(exclude_purchased),
            "query_input": query_input,
            "retrieved_items": retrieved_items,
            "answer": answer,
        }
        with open(demo_log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(turn_log, ensure_ascii=False) + "\n")
        
        print("\n" + "-"*20 + " AI Response " + "-"*20)
        print(answer)
        print("-" * 60)

except Exception as e:
    print(f"\n Error: {e}")


Dynamic GraphRAG Engine Initialized! (Ready for Inputs)



User ID (Press Enter for Cold Start, 'exit' to quit):  AEXCLMGS3Y7SRW5CMLJBNYI2HBZQ


[User Profile] Loyalty Score: 1.00 (Favorite: Dr. Denese)
   Already Purchased: 3 items


Your Question:  Find a new product similar to what I liked before, but not something I already bought.
Mode? (1: Discover New, 2: Repurchase/Loyalty) [Default: 1]:  1



Searching... (Exclude Purchased: True)


...


Searching... (Exclude Purchased: True)


Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {}, '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    // 1. Vector Search\n    // Leveraging Neo4j's native ANN index to eliminate external DB overhead and minimize latency.\n    CALL db.index.vector.queryNodes('review_embedding_index', 150, $embedding)\n    YIELD node AS similar_review, score AS vector_score\n\n    // 2. Graph Traversal\n    MATCH (similar_review)-[:EVALUATED]->(product:Product)-[:MADE_BY]->(brand:Brand)\n\n    // Optional Filter (Exclude Purchased)\n    \n\n    // 3. Personalization (Check History)\n 

Top 5 Recommendations:
   - ✨ [New Discovery] Neutrogena Hydro Boost Hyaluronic Acid H... (Score: 0.8650)
   - ✨ [New Discovery] Clinique Moisture Surge 72-Hour Auto-Rep... (Score: 0.8591)
   - ✨ [New Discovery] Naturium Multi-Peptide Moisturizer Plus ... (Score: 0.8539)

Generating Insight...

-------------------- AI Response --------------------
For your dry and sensitive skin, here are three excellent gentle moisturizers that have been highly praised for their soothing and hydrating properties:

1. **Neutrogena Hydro Boost Hyaluronic Acid Hydrating Water Gel**  
   This moisturizer is specifically formulated for dry skin and is oil-free and non-comedogenic, meaning it won't clog pores. The key ingredient, hyaluronic acid, is excellent for intense hydration without irritation. Many users with sensitive skin have found it effective and gentle, making it a strong choice for your needs.

2. **Clinique Moisture Surge 72-Hour Auto-Replenishing Hydrator**  
   Clinique is known for its all

User ID (Press Enter for Cold Start, 'exit' to quit):  exit


Exiting Demo. Goodbye! 


## Limitations & Future Work
- Benchmark against classical recommenders (e.g., MF / GNN-based CF) to quantify trade-offs.
- Enrich KG signals beyond brand loyalty (e.g., ingredients, price, category-level patterns).
